In [ ]:
# =================================================================
# CELL 0: ORTAM + VERI
# =================================================================
from pathlib import Path
import torch, os
assert torch.cuda.is_available(), 'GPU yok!'
gpu = torch.cuda.get_device_name(0)
print(gpu, '| CPU:', os.cpu_count())
if 'A100' not in gpu:
    print('!! A100 degil: 30 epoch bu GPU\'da cok uzun surer (T4\'te ~12 saat)')

!pip -q install ultralytics

from google.colab import drive
drive.mount('/content/drive')

ZIP  = '/content/drive/MyDrive/level-up-ai-roketsan-yapay-zeka-hackathonu.zip'
DATA = Path('/content/data')
if not any(DATA.rglob('annotations.csv')):
    !cp "{ZIP}" /content/data.zip
    !mkdir -p /content/data && unzip -q -o /content/data.zip -d /content/data && rm /content/data.zip
ROOT = next(DATA.rglob('annotations.csv')).parent.parent
print('veri koku:', ROOT)

NVIDIA A100-SXM4-40GB | CPU: 12
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 10.9 MB/s eta 0:00:00
Mounted at /content/drive
veri koku: /content/data


In [ ]:
# =================================================================
# CELL 1: CONFIG
# =================================================================
TRAIN_IMG  = ROOT/'train'/'images'
TEST_IMG   = ROOT/'test'/'images'
ANNOT_CSV  = ROOT/'train'/'annotations.csv'
SAMPLE_SUB = next(DATA.rglob('sample_submission.csv'))

WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)
DS   = WORK/'dataset'
RUNS = Path('/content/drive/MyDrive/yolo_runs_v3')        # Drive'da kalici
RUNS.mkdir(parents=True, exist_ok=True)

CLASSES = ['car', 'van', 'truck', 'bus']
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

# ---- Split + RFS: 11s modelleriyle AYNI (sonuclar kiyaslanabilsin) ----
SEED, BLOCK, VAL_MAX = 42, 20, 900
RFS_T = 1.0

# ---- Egitim ----
NAME     = 'm_1408_rfs_cls1_e40'
MODEL    = 'yolo11m.pt'
IMGSZ    = 1408     # egitim; test cozunurlugu Cell 5'te val'de seciliyor
EPOCHS   = 40       # patience=10 ile erken doyarsa kendiligindan durur
BATCH    = 16       # A100 40 GB. Ilk epoch'ta GPU_mem 35 GB'i gecerse 12 yap
WORKERS  = min(8, os.cpu_count())
PATIENCE = 40

AUG = dict(
    flipud=0.0,         # egik goruntu: ust=uzak, alt=yakin
    fliplr=0.5,
    degrees=0.0,        # eksen hizali kutuyu dondurmek kutuyu sisirir
    scale=0.3,          # kucuk araclari ogrenilemez boyuta indirmesin
    mosaic=1.0,
    close_mosaic=10,     # uzun egitimde mosaicsiz asama da uzun
    mixup=0.1,
    cls=1.0,            # mAP50'de siniflandirma kritik; van/truck'ta +2 puan olcmustuk
)

# ---- Inference (val'de olculmus ayarlar) ----
CONF, IOU, MAX_DET, AREA_THR = 0.001, 0.6, 1000, 180

In [ ]:
# =================================================================
# CELL 2: VERIYI YUKLE
# =================================================================
import pandas as pd, numpy as np, random, shutil, yaml, gc
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

df = pd.read_csv(ANNOT_CSV)
df.columns = [c.strip().lower() for c in df.columns]
df = df.rename(columns={'class':'label', 'category':'label', 'name':'label',
                        'xmin':'x', 'ymin':'y', 'width':'w', 'height':'h'})
df['label']    = df['label'].astype(str).str.strip().str.lower()
df['image_id'] = df['image_id'].astype(str)

id2path    = {p.stem: p for p in TRAIN_IMG.rglob('*') if p.suffix.lower() in IMG_EXT}
test_paths = sorted(p for p in TEST_IMG.rglob('*') if p.suffix.lower() in IMG_EXT)
sizes = {iid: Image.open(p).size for iid, p in tqdm(id2path.items(), desc='boyut')}
print(f'train {len(id2path)} | test {len(test_paths)} | kutu {len(df)}')

boyut:   0%|          | 0/6471 [00:00<?, ?it/s]

train 6471 | test 2118 | kutu 165349


In [ ]:
# =================================================================
# CELL 3: SPLIT + YOLO ETIKETLERI + RFS
# =================================================================
random.seed(SEED)
if DS.exists(): shutil.rmtree(DS)
for s in ('train', 'val'):
    (DS/'images'/s).mkdir(parents=True, exist_ok=True)
    (DS/'labels'/s).mkdir(parents=True, exist_ok=True)

# ---- Sahne bazli split: ardisik 20 kare tek parca train VEYA val ----
all_ids = sorted(id2path)
blocks  = [all_ids[i:i+BLOCK] for i in range(0, len(all_ids), BLOCK)]
random.shuffle(blocks)
vset = set()
for b in blocks:
    if len(vset) >= VAL_MAX: break
    vset.update(b)
train_ids = [i for i in all_ids if i not in vset]
val_ids   = sorted(vset)
assert not (set(train_ids) & set(val_ids))
print(f'train {len(train_ids)} | val {len(val_ids)}')
if len(val_ids) != 900:
    print('!! val 900 degil: 11s sonuclariyla karsilastirma gecersiz olur')

# ---- YOLO formati ----
cls_idx = {c: i for i, c in enumerate(CLASSES)}
by_img  = dict(tuple(df.groupby('image_id')))

def write_split(ids, split):
    for iid in tqdm(ids, desc=split):
        src = id2path[iid]; W, H = sizes[iid]
        (DS/'images'/split/f'{iid}{src.suffix.lower()}').symlink_to(src)
        lines = []
        for r in by_img.get(iid, df.iloc[:0]).itertuples():
            if r.label not in cls_idx: continue
            x1, y1 = max(0., r.x), max(0., r.y)
            x2, y2 = min(W, r.x + r.w), min(H, r.y + r.h)
            if x2 - x1 < 1 or y2 - y1 < 1: continue
            lines.append(f'{cls_idx[r.label]} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} '
                         f'{(x2-x1)/W:.6f} {(y2-y1)/H:.6f}')
        (DS/'labels'/split/f'{iid}.txt').write_text('\n'.join(lines))   # bos = arac yok

write_split(train_ids, 'train')
write_split(val_ids, 'val')

# ---- RFS: nadir sinif iceren goruntuler tekrarlanir (sadece train) ----
img_cls = df[df.image_id.isin(train_ids)].groupby('image_id').label.agg(set)
freq  = {c: sum(c in s for s in img_cls) / len(train_ids) for c in CLASSES}
r_cls = {c: max(1.0, np.sqrt(RFS_T / freq[c])) for c in CLASSES}
rng = np.random.default_rng(SEED)
train_list = []
for iid in train_ids:
    p = DS/'images'/'train'/f'{iid}{id2path[iid].suffix.lower()}'
    r = max((r_cls[c] for c in img_cls.get(iid, set())), default=1.0)
    train_list += [str(p)] * (int(r) + int(rng.random() < r - int(r)))

(WORK/'train_list.txt').write_text('\n'.join(train_list))
YAML = WORK/'data.yaml'
YAML.write_text(yaml.safe_dump({'path': str(DS), 'train': str(WORK/'train_list.txt'),
    'val': 'images/val', 'names': dict(enumerate(CLASSES))}, sort_keys=False))

print(f'RFS: {len(train_ids)} -> {len(train_list)} satir')
print('r(c):', {c: round(v, 2) for c, v in r_cls.items()})

train 5571 | val 900


train:   0%|          | 0/5571 [00:00<?, ?it/s]

val:   0%|          | 0/900 [00:00<?, ?it/s]

RFS: 5571 -> 7885 satir
r(c): {'car': np.float64(1.03), 'van': np.float64(1.15), 'truck': np.float64(1.36), 'bus': np.float64(1.83)}


In [ ]:
# =================================================================
# CELL 4: EGITIM (A100, ~2.5-3 saat)
# Colab koparsa Cell 0-3'u calistir, sonra bunu: kaldigi yerden devam eder.
# =================================================================
from ultralytics.utils import metrics as um
um.Metric.fitness = lambda self: float(self.map50)   # best.pt = en yuksek mAP50

run_dir = RUNS/NAME
last    = run_dir/'weights'/'last.pt'

if last.exists():
    try:
        print('>> yarim kalmis, devam ediliyor')
        YOLO(last).train(resume=True)
    except AssertionError:
        print('>> zaten bitmis')
else:
    YOLO(MODEL).train(
        data=str(YAML), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, device=0,
        project=str(RUNS), name=NAME, exist_ok=True, seed=SEED,
        optimizer='SGD', lr0=0.01, warmup_epochs=3, cos_lr=True,
        patience=PATIENCE, save_period=3,
        workers=WORKERS, amp=True, cache='disk', plots=True,
        **AUG)

W_M = run_dir/'weights'/'best.pt'

res = pd.read_csv(run_dir/'results.csv'); res.columns = res.columns.str.strip()
col = 'metrics/mAP50(B)'
print(f'en yuksek val mAP50: {res[col].max():.4f} (epoch {int(res[col].idxmax()) + 1}/{len(res)})')
print('son 5 epoch:', res[col].tail(5).round(4).tolist())

Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=1.0, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/work/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1408, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=m_1408_rfs_cls1_e40, nbs=64, nms=None, op

In [ ]:
# =================================================================
# HUCRE 1: KURULUM (veri, val seti, multi-label, yardimci fonksiyonlar)
# =================================================================
!pip -q install ultralytics ensemble-boxes

import torch, gc, random, warnings, psutil
import numpy as np, pandas as pd
from pathlib import Path
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion
import ultralytics.utils.ops as _ops
try:
    import ultralytics.utils.nms as _nms
except ImportError:
    _nms = None
warnings.filterwarnings('ignore', message='Zero area box')
assert torch.cuda.is_available(), 'GPU yok!'

# ---- Ayarlar ----
RUNS = Path('/content/drive/MyDrive/yolo_runs_v3')
MODELS = {
    'm_1280': RUNS/'m_1280_rfs_cls1'/'weights'/'best.pt',
    'm_1408': RUNS/'m_1408_rfs_cls1_e40'/'weights'/'best.pt',
}
ZIP      = '/content/drive/MyDrive/level-up-ai-roketsan-yapay-zeka-hackathonu.zip'
SIZES    = (1408, 1536, 1664)
CLASSES  = ['car', 'van', 'truck', 'bus']
SEED, BLOCK, VAL_MAX = 42, 20, 900
NMS_IOU, WBF_THR, AREA_THR = 0.6, 0.65, 180
BATCH    = 8
IMG_EXT  = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

# ---- Multi-label: kararsiz araclarda iki sinif tahmini de tutulur ----
_mods = [m for m in (_ops, _nms) if m is not None and hasattr(m, 'non_max_suppression')]
for _m in _mods:
    if not hasattr(_m.non_max_suppression, '_orig'):
        _o = _m.non_max_suppression
        def _f(*a, _o=_o, **k):
            k['multi_label'] = True
            return _o(*a, **k)
        _f._orig = _o
        _m.non_max_suppression = _f
assert all(hasattr(m.non_max_suppression, '_orig') for m in _mods), 'multi-label yok'

# ---- Veri ----
from google.colab import drive
drive.mount('/content/drive')
for k, w in MODELS.items():
    assert w.exists(), f'{k} agirligi yok: {w}'

DATA = Path('/content/data')
if not any(DATA.rglob('annotations.csv')):
    !mkdir -p /content/data && unzip -q -o "{ZIP}" -d /content/data
ROOT = next(DATA.rglob('annotations.csv')).parent.parent
SAMPLE_SUB = next(DATA.rglob('sample_submission.csv'))

df = pd.read_csv(ROOT/'train'/'annotations.csv')
df.columns = [c.strip().lower() for c in df.columns]
df = df.rename(columns={'class':'label', 'category':'label', 'name':'label',
                        'xmin':'x', 'ymin':'y', 'width':'w', 'height':'h'})
df['label'] = df['label'].astype(str).str.strip().str.lower()
df['image_id'] = df['image_id'].astype(str)
id2path = {p.stem: p for p in (ROOT/'train'/'images').rglob('*') if p.suffix.lower() in IMG_EXT}
test_paths = sorted(p for p in (ROOT/'test'/'images').rglob('*') if p.suffix.lower() in IMG_EXT)

# ---- Val seti (egitimdekiyle ayni) ----
random.seed(SEED)
all_ids = sorted(id2path)
blocks = [all_ids[i:i+BLOCK] for i in range(0, len(all_ids), BLOCK)]
random.shuffle(blocks)
vset = set()
for b in blocks:
    if len(vset) >= VAL_MAX: break
    vset.update(b)
val_ids = sorted(vset)
assert len(val_ids) == 900

# .txt listeleri: goruntuler tek tek okunur, RAM dolmaz
VAL_TXT, TEST_TXT = Path('/content/val_list.txt'), Path('/content/test_list.txt')
VAL_TXT.write_text('\n'.join(str(id2path[i]) for i in val_ids))
TEST_TXT.write_text('\n'.join(str(p) for p in test_paths))
print(f'val {len(val_ids)} | test {len(test_paths)}')

# ---- Gercek kutular + mAP@0.5 ----
G = df[df.image_id.isin(val_ids)].copy()
G['cls'] = G.label.map({c: i for i, c in enumerate(CLASSES)})
GTD = {c: {i: np.c_[d.x, d.y, d.x + d.w, d.y + d.h].astype(float)
           for i, d in G[G.cls == c].groupby('image_id')} for c in range(4)}
NGT = {c: int((G.cls == c).sum()) for c in range(4)}

def iou_mat(a, b):
    iw = np.clip(np.minimum(a[:, None, 2], b[:, 2]) - np.maximum(a[:, None, 0], b[:, 0]), 0, None)
    ih = np.clip(np.minimum(a[:, None, 3], b[:, 3]) - np.maximum(a[:, None, 1], b[:, 1]), 0, None)
    inter = iw * ih
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + bb - inter + 1e-9)

def voc_ap(rec, prec):
    mrec = np.r_[0, rec, 1]; mpre = np.r_[0, prec, 0]
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[i+1] - mrec[i]) * mpre[i+1]))

def evaluate(res):
    """res: {iid: (kutular_piksel, skorlar, siniflar)} -> sinif bazli AP ve mAP50"""
    aps = {}
    for c, name in enumerate(CLASSES):
        S, T = [], []
        for iid, (b, s, l) in res.items():
            k = l == c
            if not k.any(): continue
            b, s = b[k], s[k]; o = np.argsort(-s); b, s = b[o], s[o]
            tp = np.zeros(len(s)); gt = GTD[c].get(iid)
            if gt is not None:
                M = iou_mat(b, gt); used = np.zeros(len(gt), bool)
                for j in range(len(s)):
                    g = M[j].argmax()
                    if M[j, g] >= 0.5 and not used[g]:
                        tp[j] = 1; used[g] = True
            S.append(s); T.append(tp)
        s, tp = np.concatenate(S), np.concatenate(T)
        tp = tp[np.argsort(-s, kind='stable')]
        ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp)
        aps[name] = voc_ap(ctp / max(NGT[c], 1), ctp / np.maximum(ctp + cfp, 1e-9))
    aps['mAP50'] = np.mean([aps[c] for c in CLASSES])
    return pd.Series(aps).round(4)

# ---- Tahmin ve WBF ----
_loaded = {}
def predict(key, sz, txt):
    """Bir modelin bir boyuttaki tahminleri: {iid: (0-1 kutular, skor, sinif)}, {iid: (W, H)}"""
    if key not in _loaded:
        _loaded[key] = YOLO(MODELS[key])
        assert list(_loaded[key].names.values()) == CLASSES
    d, shapes = {}, {}
    for r in _loaded[key].predict(str(txt), imgsz=sz, conf=0.001, iou=NMS_IOU, max_det=1000,
                                  device=0, batch=BATCH, stream=True, verbose=False):
        iid = Path(r.path).stem
        H, W = r.orig_shape; shapes[iid] = (W, H); b = r.boxes
        d[iid] = (np.clip(b.xyxy.cpu().numpy() / [W, H, W, H], 0, 1),
                  b.conf.cpu().numpy(), b.cls.cpu().numpy())
    gc.collect()
    return d, shapes

def fuse(preds, shapes, weights):
    """WBF -> {iid: (piksel kutular, skor, sinif)}, alan filtresi uygulanmis"""
    empty = (np.zeros((0, 4)), np.zeros(0), np.zeros(0))
    res = {}
    for iid, (W, H) in shapes.items():
        B, S, L = zip(*[p.get(iid, empty) for p in preds])
        b, s, l = weighted_boxes_fusion([x.tolist() for x in B], [x.tolist() for x in S],
                                        [x.tolist() for x in L], weights=weights,
                                        iou_thr=WBF_THR, skip_box_thr=0.001, conf_type='avg')
        b = b * [W, H, W, H]
        k = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]) >= AREA_THR
        res[iid] = (b[k], s[k], l[k].astype(int))
    return res

print('kurulum tamam')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
val 900 | test 2118
kurulum tamam


In [ ]:
# =================================================================
# HUCRE 2: IKI MODEL + ENSEMBLE, VAL'DE KARSILASTIR
# =================================================================
VP, vshape = {}, None
for key in MODELS:
    for sz in SIZES:
        VP[(key, sz)], vshape = predict(key, sz, VAL_TXT)
        print(f'val tahmini: {key} @ {sz}')

# Secenekler: [(model, boyut, WBF agirligi), ...]
OPTIONS = {
    'm_1280 tek':         [('m_1280', s, 1) for s in SIZES],
    'm_1408 tek':         [('m_1408', s, 1) for s in SIZES],
    'ensemble esit':      [('m_1280', s, 1) for s in SIZES] + [('m_1408', s, 1) for s in SIZES],
    'ensemble 1408 agir': [('m_1280', s, 1) for s in SIZES] + [('m_1408', s, 2) for s in SIZES],
    'ensemble 1280 agir': [('m_1280', s, 2) for s in SIZES] + [('m_1408', s, 1) for s in SIZES],
}

log = {}
for name, cfg in OPTIONS.items():
    res = fuse([VP[(k, s)] for k, s, _ in cfg], vshape, [w for _, _, w in cfg])
    log[name] = evaluate(res)
    print(f'{name:22s} mAP50 {log[name].mAP50:.4f}')

tab = pd.DataFrame(log).T.sort_values('mAP50', ascending=False)
tab['m_1280e_gore'] = (tab.mAP50 - log['m_1280 tek'].mAP50).round(4)
print('\n', tab.to_string())

# Karar: m_1280'i en az 0.005 gecen yoksa m_1280'de kal
BEST = tab.index[0] if tab.iloc[0].m_1280e_gore >= 0.005 else 'm_1280 tek'
CONFIGS = OPTIONS[BEST]
print(f'\n>> SECILEN: {BEST} | val mAP50 {log[BEST].mAP50:.4f}')

del VP; gc.collect()

val tahmini: m_1280 @ 1408
val tahmini: m_1280 @ 1536
val tahmini: m_1280 @ 1664
val tahmini: m_1408 @ 1408
val tahmini: m_1408 @ 1536


KeyboardInterrupt: 

In [ ]:
# =================================================================
# ENSEMBLE + TTA, VAL'DE (Hucre 1 calismis olmali)
# =================================================================
def predict_aug(key, sz, txt, augment):
    """predict ile ayni, TTA secenekli."""
    if key not in _loaded:
        _loaded[key] = YOLO(MODELS[key])
    d, shapes = {}, {}
    for r in _loaded[key].predict(str(txt), imgsz=sz, conf=0.001, iou=NMS_IOU, max_det=1000,
                                  augment=augment, device=0, batch=BATCH,
                                  stream=True, verbose=False):
        iid = Path(r.path).stem
        H, W = r.orig_shape; shapes[iid] = (W, H); b = r.boxes
        d[iid] = (np.clip(b.xyxy.cpu().numpy() / [W, H, W, H], 0, 1),
                  b.conf.cpu().numpy(), b.cls.cpu().numpy())
    gc.collect()
    return d, shapes

# ---- Tahminler: normal (6) + TTA 1536 (2) ----
VP, vshape = {}, None
for key in MODELS:
    for sz in SIZES:
        VP[(key, sz, False)], vshape = predict_aug(key, sz, VAL_TXT, False)
        print(f'{key} @ {sz}')
    VP[(key, 1536, True)], _ = predict_aug(key, 1536, VAL_TXT, True)
    print(f'{key} @ 1536 TTA')

# ---- Secenekler ----
normal   = [(k, s, False) for k in MODELS for s in SIZES]
tta_ekle = normal + [(k, 1536, True) for k in MODELS]
tta_degis = [(k, s, s == 1536) for k in MODELS for s in SIZES]

log = {}
for name, cfg in [('mevcut ensemble (6)', normal),
                  ('TTA ekle (8)', tta_ekle),
                  ('TTA ile degistir (6)', tta_degis)]:
    log[name] = evaluate(fuse([VP[c] for c in cfg], vshape, [1] * len(cfg)))
    print(f'{name:22s} mAP50 {log[name].mAP50:.4f}')

tab = pd.DataFrame(log).T
tab['fark'] = (tab.mAP50 - tab.loc['mevcut ensemble (6)', 'mAP50']).round(4)
print('\n', tab.to_string())

m_1280 @ 1408
m_1280 @ 1536
m_1280 @ 1664
m_1280 @ 1536 TTA
m_1408 @ 1408
m_1408 @ 1536
m_1408 @ 1664
m_1408 @ 1536 TTA
mevcut ensemble (6)    mAP50 0.8408
TTA ekle (8)           mAP50 0.8435
TTA ile degistir (6)   mAP50 0.8430

                          car     van   truck     bus   mAP50    fark
mevcut ensemble (6)   0.9233  0.7440  0.8229  0.8729  0.8408  0.0000
TTA ekle (8)          0.9236  0.7472  0.8259  0.8771  0.8435  0.0027
TTA ile degistir (6)  0.9236  0.7468  0.8253  0.8766  0.8430  0.0022


In [ ]:
# =================================================================
# HUCRE 3: SECILEN AYARLA TEST SUBMISSION
# =================================================================
TP, tshape = [], None
for key, sz, _ in CONFIGS:
    d, tshape = predict(key, sz, TEST_TXT)
    TP.append(d)
    print(f'test tahmini: {key} @ {sz} | RAM %{psutil.virtual_memory().percent:.0f}')

res = fuse(TP, tshape, [w for _, _, w in CONFIGS])

# ---- Format: "sinif skor x y w h ..." ----
names = dict(enumerate(CLASSES))
pred_str = {}
for iid, (b, s, l) in res.items():
    wh = b[:, 2:] - b[:, :2]
    pred_str[iid] = ' '.join(
        f'{names[l[i]]} {s[i]:.4f} {int(round(b[i, 0]))} {int(round(b[i, 1]))} '
        f'{int(round(wh[i, 0]))} {int(round(wh[i, 1]))}' for i in np.argsort(-s))

# ---- Kontroller ----
assert len(pred_str) == len(test_paths), 'bazi goruntuler islenmedi'
sub_tmpl = pd.read_csv(SAMPLE_SUB); sub_tmpl['image_id'] = sub_tmpl['image_id'].astype(str)
sub = sub_tmpl[['image_id']].copy()
sub['PredictionString'] = sub.image_id.map(pred_str).replace('', np.nan).fillna('none')
assert set(sub.image_id) <= set(pred_str), "sablondaki bazi id'ler yok"

bad = 0
for s_ in sub.PredictionString:
    if s_ == 'none': continue
    p = s_.split()
    if len(p) % 6: bad += 1; continue
    for i in range(0, len(p), 6):
        if p[i] not in CLASSES or not 0 <= float(p[i+1]) <= 1 or int(p[i+4]) <= 0 or int(p[i+5]) <= 0:
            bad += 1; break
assert bad == 0, f'{bad} hatali satir'

# ---- Kayit ----
OUT = RUNS/f"submission_{BEST.replace(' ', '_')}.csv"
n = sub.PredictionString.map(lambda x: 0 if x == 'none' else len(x.split()) // 6)
sub.to_csv(OUT, index=False)
print(f'\n{BEST} | none {(n == 0).sum()} | kutu ort {n.mean():.0f} max {n.max()} | '
      f'{OUT.stat().st_size/1e6:.1f} MB -> {OUT}')
print('ornek:', sub.iloc[0].PredictionString[:100], '...')

from google.colab import files
files.download(str(OUT))

test tahmini: m_1280 @ 1408 | RAM %18
test tahmini: m_1280 @ 1536 | RAM %18
test tahmini: m_1280 @ 1664 | RAM %18
test tahmini: m_1408 @ 1408 | RAM %18
test tahmini: m_1408 @ 1536 | RAM %18
test tahmini: m_1408 @ 1664 | RAM %18

ensemble esit | none 0 | kutu ort 273 max 2006 | 14.5 MB -> /content/drive/MyDrive/yolo_runs_v3/submission_ensemble_esit.csv
ornek: car 0.9104 531 475 69 84 van 0.8923 506 242 59 57 truck 0.8883 1013 277 149 73 car 0.8764 557 365 47 ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>